# Day 5 — Cross-Validation & Error Analysis
**Project:** Tehran House Price Prediction

## Goal
Check whether the Day 4 baseline results are reliable, then inspect where the best model makes large errors.

**Tasks tonight**
1. Run every cell from top to bottom.
2. Check the cross-validation and test-set tables.
3. Inspect the residual and actual-vs-predicted plots.
4. Fill in the short conclusion at the end.

Do not tune model parameters yet. Keep the test set separate from cross-validation.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tehran_house_prices_cleaned.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}" 

## 2. Load data and define features
We exclude derived price columns to avoid target leakage.

In [ ]:
df = pd.read_csv(DATA_PATH)

TARGET = "Price"
NUMERIC_FEATURES = ["Area", "Room"]
BOOLEAN_FEATURES = ["Parking", "Warehouse", "Elevator"]
CATEGORICAL_FEATURES = ["Address"]
FEATURES = NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES

required = FEATURES + [TARGET]
missing_columns = [c for c in required if c not in df.columns]
assert not missing_columns, f"Missing columns: {missing_columns}"

df = df[required].copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0].copy()
df["Address"] = df["Address"].fillna("Unknown").astype(str)

X = df[FEATURES]
y = df[TARGET]

print("Rows used:", len(df))
print("Feature columns:", FEATURES)
print("Target: total house price in rials")
display(df.head())

## 3. Make the fixed train/test split
The test set is held aside for final evaluation. Cross-validation uses only the training set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 4. Build preprocessing and baseline models

In [ ]:
numeric_transformer = SimpleImputer(strategy="median")

# Convert Boolean values to numeric BEFORE imputation.
boolean_transformer = Pipeline(steps=[
    ("to_numeric", FunctionTransformer(
        lambda x: x.astype("float64"), feature_names_out="one-to-one"
    )),
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("bool", boolean_transformer, BOOLEAN_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES)
])

models = {
    "Dummy median baseline": DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42, min_samples_leaf=5
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, min_samples_leaf=2, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42, n_estimators=100, learning_rate=0.05, max_depth=2
    ),
}

print("Models ready:", ", ".join(models.keys()))

## 5. Five-fold cross-validation on training data
Each fold uses a different part of the training data for validation. Lower MAE/RMSE is better; higher R² is better.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

cv_rows = []
for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=1, return_train_score=False
    )
    cv_rows.append({
        "Model": name,
        "CV_MAE_rials": -scores["test_mae"].mean(),
        "CV_RMSE_rials": -scores["test_rmse"].mean(),
        "CV_R2": scores["test_r2"].mean(),
        "CV_RMSE_std": scores["test_rmse"].std(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV_RMSE_rials").reset_index(drop=True)
display(cv_results)
cv_results.to_csv(REPORTS_DIR / "day5_cross_validation_results.csv", index=False)
print("Saved:", REPORTS_DIR / "day5_cross_validation_results.csv")

## 6. Fit models on all training data and evaluate once on the test set

In [ ]:
fitted_models = {}
test_predictions = {}
test_rows = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted_models[name] = pipe
    test_predictions[name] = pred
    test_rows.append({
        "Model": name,
        "Test_MAE_rials": mean_absolute_error(y_test, pred),
        "Test_RMSE_rials": mean_squared_error(y_test, pred) ** 0.5,
        "Test_R2": r2_score(y_test, pred),
    })

test_results = pd.DataFrame(test_rows).sort_values("Test_RMSE_rials").reset_index(drop=True)
display(test_results)
test_results.to_csv(REPORTS_DIR / "day5_test_results.csv", index=False)
print("Saved:", REPORTS_DIR / "day5_test_results.csv")

## 7. Compare cross-validation and test RMSE

In [ ]:
comparison = cv_results[["Model", "CV_RMSE_rials"]].merge(
    test_results[["Model", "Test_RMSE_rials"]], on="Model"
).sort_values("CV_RMSE_rials")

display(comparison)

plot_data = comparison.set_index("Model")[["CV_RMSE_rials", "Test_RMSE_rials"]]
ax = plot_data.plot(kind="barh", figsize=(10, 5))
ax.set_xlabel("RMSE (rials) — lower is better")
ax.set_title("Cross-validation vs test-set RMSE")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "day5_cv_vs_test_rmse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", FIGURES_DIR / "day5_cv_vs_test_rmse.png")

## 8. Error analysis for the best test-set model
Residual = actual price − predicted price. Large absolute residuals mean large prediction errors.

In [ ]:
best_model_name = test_results.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_pred = test_predictions[best_model_name]

errors = pd.DataFrame({
    "Actual_Price_rials": y_test.to_numpy(),
    "Predicted_Price_rials": best_pred,
})
errors["Residual_rials"] = errors["Actual_Price_rials"] - errors["Predicted_Price_rials"]
errors["Absolute_Error_rials"] = errors["Residual_rials"].abs()
errors["Absolute_Percent_Error"] = (
    errors["Absolute_Error_rials"] / errors["Actual_Price_rials"].clip(lower=1) * 100
)

print("Best model by test RMSE:", best_model_name)
print("Test MAE (rials):", round(mean_absolute_error(y_test, best_pred), 2))
print("Test RMSE (rials):", round(mean_squared_error(y_test, best_pred) ** 0.5, 2))
print("Test R²:", round(r2_score(y_test, best_pred), 4))

display(errors.sort_values("Absolute_Error_rials", ascending=False).head(10))
errors.to_csv(REPORTS_DIR / "day5_best_model_errors.csv", index=False)
print("Saved:", REPORTS_DIR / "day5_best_model_errors.csv")

## 9. Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(errors["Actual_Price_rials"], errors["Predicted_Price_rials"], alpha=0.6)
low = min(errors["Actual_Price_rials"].min(), errors["Predicted_Price_rials"].min())
high = max(errors["Actual_Price_rials"].max(), errors["Predicted_Price_rials"].max())
axes[0].plot([low, high], [low, high], linestyle="--")
axes[0].set_xlabel("Actual price (rials)")
axes[0].set_ylabel("Predicted price (rials)")
axes[0].set_title(f"Actual vs predicted — {best_model_name}")

axes[1].scatter(errors["Predicted_Price_rials"], errors["Residual_rials"], alpha=0.6)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted price (rials)")
axes[1].set_ylabel("Residual (actual − predicted)")
axes[1].set_title("Residuals vs predicted")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "day5_error_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", FIGURES_DIR / "day5_error_diagnostics.png")

## 10. Day 5 conclusions — complete these tasks
Write your answers in the markdown cell below after checking the outputs.

- Best model by cross-validation RMSE:
- Best model by test RMSE:
- Is the CV RMSE reasonably close to the test RMSE? (yes/no + one sentence)
- What pattern do you notice in the residual plot?
- What should be improved next: features, data quality, or model tuning? Give one reason.

**Reminder:** A good R² alone does not mean every prediction is accurate. Look at the errors and the price scale.

**My Day 5 answers**

- Best model by cross-validation RMSE: _fill in_
- Best model by test RMSE: _fill in_
- CV vs test comparison: _fill in_
- Residual plot observation: _fill in_
- Next improvement and reason: _fill in_